# **02477 Bayesian Machine Learning | Comprehensive study notes**



---

###  **Table of Contents**




1. [Week 1 | foundations & beta-binomial model](#week1)

2. [Week 2 | predictions, plug-in & grid approximations](#week2)

3. [Week 3 | bayesian classification & laplace approximations](#week3)

4. [Week 4 | bayesian linear regression](#week4)

5. [Week 5 | gaussian processes (regression)](#week5)

6. [Week 6 | GP classification & kernels](#week6)



---

<a id='week1'></a>

<div class="alert alert-block alert-info">

### **Week 1 | foundations & beta-binomial model**

Bayesian machine learning, the binominal model, maximum likelihood estimation (MLE), bayesian inference, beta-binomial model
</div>

##### **1.1 Why bayesian machine learning?**


Classical ML picks **one** parameter setting and makes predictions with it. Bayesian ML **averages over all parameter settings** weighted by their posterior probability:

$$p(y^*|y) = \int p(y^*|\theta)\, p(\theta|y)\, d\theta$$

The slides uses the following notation for "making predictions using a weighted average of all possible parameter sets" as $y^* = \sum_{i=1}^M f(x^*|w_i)p(w_i|y)$ and "with infinitely many parameter settings" as $y^* = \int f(x^*|w)p(w|y)dw$.

**Advantages:**

> Principled uncertainty quantification (epistemic + aleatoric)

> Less prone to overfitting (prior regularises)

> Natural model selection via marginal likelihood

> Better decision-making under uncertainty


Bayesian inference for supervised learning has workflow of first, having $D = \left\{ x_i, y:i \right\}_{i=1}^N$ and $p(w,y) = p(y|w)p(w)$, which are used to compute poasterior $p(w|y)$, which is then used to compute predictive $p(y^*|x^*,y)$ which also feeds on $x^*$. Finally, these are used to summarize results using mean, mode and intervals.

(this is actually from week 3)

##### **1.2 Types of uncertainty**




| Type | Cause | Reducible? |
|------|-------|-----------|
| **Epistemic** | Lack of data / knowledge | Yes - more data helps |
| **Aleatoric** | Inherent noise in the process | No - irreducible |


##### **1.3 Bayes' Rule - the cornerstone**


This is a systematic way to combine data with prior knowledge:

<span style="color: blue;">


$$\underbrace{p(\theta | y)}_{\text{posterior}} = \frac{\underbrace{p(y|\theta)}_{\text{likelihood}}\; \underbrace{p(\theta)}_{\text{prior}}}{\underbrace{p(y)}_{\text{evidence/marginal likelihood}}}$$

</span>

<span style="color: red;">

Where $\theta$ is the unknown parameter and $y$ is observed data.

</span>

The practical recipe is:

> What have we observed? This goes after the $|$ everywhere

> What are we uncertain about? This goes before the $|$ on the left

> Numerator is the joint distribution of uncertain things given observed things

> Denominator is the marginal of just the data given observed inputs (normalisation constant)


| Term | Symbol | Meaning |
|------|--------|---------|
| Prior | $p(\theta)$ | Belief about $\theta$ **before** seeing data |
| Likelihood | $p(y\|\theta)$ | Probability of data **given** parameters |
| Posterior | $p(\theta\|y)$ | Updated belief **after** seeing data |
| Evidence | $p(y) = \int p(y\|\theta)p(\theta)d\theta$ | Normalisation constant; useful for model selection and for Beta binomial it is $p(y) = \binom{N}{y} \frac{B(y+\alpha_0, N-y+\beta_0)}{B(\alpha_0, \beta_0)}$ |

**Key insight:**
$$p(\theta|y) \propto p(y|\theta)\,p(\theta)$$

We never need to compute $p(y)$ to find the posterior shape, since $p(y)$ only changes the expression with a constant, as it is alsways a factor, since $\theta$ is integrated out (so $p(y)$ is not needed for finding MAP estimate, drawing posterior samples from MCMC as the MH acceptance ratio cancels it out).

##### **1.4 Common distributions (building blocks)**


| Distribution | Domain | PMF / PDF | Mean | Variance | Examples |
|---|---|---|---|---|---|
| $\text{Bernoulli}(\mu)$ | $\{0,1\}$ | $\mu^y(1-\mu)^{1-y}$ | $\mu$ | $\mu(1-\mu)$ | Binary data|
| $\text{Binomial}(N,\theta)$ | $\{0,\dots,N\}$ | $\binom{N}{y}\theta^y(1-\theta)^{N-y}$ | $N\theta$ | $N\theta(1-\theta)$ | $k$ out of $N$ successes|
| $\text{Beta}(a,b)$ | $[0,1]$ | $\propto\theta^{a-1}(1-\theta)^{b-1}$ | $\frac{a}{a+b}$ | $\frac{ab}{(a+b)^2(a+b+1)}$ | Reasoning about probabilities |
| $\mathcal{N}(\mu,\sigma^2)$ | $\mathbb{R}$ | $\frac{1}{\sqrt{2\pi\sigma^2}}\exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$ | $\mu$ | $\sigma^2$ | Gaussian distribution of real numbers |
| $\text{Poisson}(\lambda)$ | $\{0,1,2,\dots\}$ | $e^{-\lambda}\lambda^y/y!$ | $\lambda$ | $\lambda$ | Count data |
| $\text{Categorical}(\pi)$ | $\{1,\dots,K\}$ | $\prod_k \pi_k^{[y=k]}$ | - | - | Multi-class data |
| $\text{Gamma}(\alpha,\beta)$ | $\mathbb{R}^+$ | $\frac{\beta^\alpha}{\Gamma(\alpha)} x^{\alpha-1} e^{-\beta x}$ | $\frac{\alpha}{\beta}$ | $\frac{\alpha}{\beta^2}$ | Positive real times and numbers |
| $\text{Dirichlet}(\alpha)$ | $\{\pi_k \geq 0, \sum_k \pi_k = 1\}$ | $\frac{1}{B(\alpha)}\prod_k \pi_k^{\alpha_k - 1}$ | $\frac{\alpha_k}{\sum_j \alpha_j}$ | $\frac{\alpha_k(\sum_j \alpha_j - \alpha_k)}{(\sum_j \alpha_j)^2 (\sum_j \alpha_j + 1)}$ | Reasoning about probability vectors |

These include the Bernoulli distribution, Binomial distribution, Beta distribution, Gaussian Normal distribution, Poisson distribution and Categorical distribution.


**Univariate** $x \in \mathbb{R}$:

<span style="color: blue;">

$$\mathcal{N}(x|\mu,\sigma^2) = \frac{1}{\sqrt{2\pi\sigma^2}}\exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)$$

</span>

**Multivariate** $\boldsymbol{x} \in \mathbb{R}^D$:

<span style="color: blue;">

$$p(\boldsymbol{x} | \boldsymbol{\mu}, \Sigma) = \frac{1}{(2\pi)^{D/2} |\Sigma|^{1/2}} \exp\!\left( -\frac{1}{2} (\boldsymbol{x}-\boldsymbol{\mu})^T \Sigma^{-1} (\boldsymbol{x}-\boldsymbol{\mu}) \right)$$

</span>

<span style="color: red;">

Where $\boldsymbol{\mu}$ is the mean vector, $\Sigma$ is the covariance matrix, $D$ is the dimensionality, and $|\Sigma|$ is the determinant. The term $(\boldsymbol{x}-\boldsymbol{\mu})^T \Sigma^{-1} (\boldsymbol{x}-\boldsymbol{\mu})$ is the **Mahalanobis distance** - the multivariate generalisation of "how many standard deviations is $\boldsymbol{x}$ from $\boldsymbol{\mu}$?". When $\Sigma = I$ it reduces to squared Euclidean distance.

</span>



**Log density (log-likelihood)** (used to identify Gaussians by pattern-matching):

<span style="color: blue;">

$$\ln \mathcal{N}(x|\mu,\sigma^2) = -\frac{1}{2\sigma^2}x^2 + \frac{\mu}{\sigma^2}x + K$$
$$\ln \mathcal{N}(w|m,S) = -\frac{1}{2}w^T S^{-1}w + m^T S^{-1}w + K$$

</span>

THIS FORMULA ASSUMES A FIXED CONSTANT FOR THE MEAN, AND DONT USE THIS FORMULA FULLY BECAUSE IT ABSORBS ALOT TO BE IN A CONSTANT $K$. SO IF YOU NEED A NUMERICAL VALUE, USE `mvn.logpdf` AND IF $\mu = \mu(\theta)$ so like it depends on some parameter like in 2024R, Q5.3, then we write:
$$
\ln \mathcal{N}(y | \mu(\theta), \sigma^2) = -\frac{1}{2\sigma^2 }(y-\mu(\theta))^2 + K
$$

> **Key insight:** Any distribution whose log density is a quadratic in the variable is Gaussian. Match coefficients of the quadratic and linear terms to read off the mean and covariance.



Some important properties:

**Affine transformations** (scalar): $x \sim \mathcal{N}(m,v) \Rightarrow a+bx \sim \mathcal{N}(a+bm,\, b^2v)$

**Affine transformations** (vector): $x \sim \mathcal{N}(m_x, V_x) \Rightarrow a+Bx \sim \mathcal{N}(a+Bm_x,\, BV_xB^T)$

**Sum of independents**: $x \perp y \Rightarrow x+y \sim \mathcal{N}(m_x+m_y,\, V_x+V_y)$

<span style="color: red;">

**How to apply these to any exam question:**

For a linear system like $z = Ax + b + n_1$ where $x$ is **observed** (conditioning on it):

> The observed variable $x$ becomes a **fixed constant** — it is no longer random

> Rewrite as $z = \underbrace{Ax + b}_{\text{this is } a, \text{ a constant}} + \underbrace{n_1}_{\text{this is the random part}}$

> Now it matches $a + Bx$ with $a = Ax+b$, $B = I$, $x = n_1$

> Apply sum of independents: mean = $Ax + b + \mathbb{E}[n_1] = Ax + b$, covariance = $\text{Cov}(n_1)$

$$p(z|x) = \mathcal{N}(z \mid Ax + b,\ \text{Cov}(n_1))$$

**The covariance is always just the covariance of the noise term.** Read it directly from the problem statement.

**What changes when you do NOT condition on $x$** (i.e. finding $p(z)$):

> Now $x$ is also random, so both $Ax$ and $n_1$ contribute to the variance

> Apply affine transformation to $Ax$: covariance becomes $A\,\text{Cov}(x)\,A^T$

> Then add noise: total covariance = $A\,\text{Cov}(x)\,A^T + \text{Cov}(n_1)$

$$p(z) = \mathcal{N}(z \mid Ab + \mathbb{E}[b],\ A\,\text{Cov}(x)\,A^T + \text{Cov}(n_1))$$

</span>

##### **1.4.5 More about Beta distribution**


MORE about beta distribution from the slides is that is it a family of distributions for a random variable $\theta \in [0,1]$, and density of the Beta distribution is given by (and it integrates to $1$):
$$
p(\theta | a,b) = \frac{1}{B(a,b)} \theta^{a-1}(1-\theta)^{b-1}
$$
With the $B(a,b)$ being normalization constant given by:
$$
B(a,b) = \frac{\Gamma(a)\Gamma(b)}{\Gamma(a+b)}
$$
With:
$$
\Gamma(z) = \int_0^\infty t^{z-1} e^{-t} dt
$$
Functional form of Beta density is:
$$
f(\theta) = \theta^{a-1} (1-\theta)^{b-1}
$$
**Special case:** $\text{Beta}(1,1)$ is the uniform distribution on $[0,1]$, commonly used as non-informative prior.

##### **1.5 The beta-binomial model (conjugate)**



**Setup:** coin with unknown bias $\theta \in [0,1]$, observe $y$ heads in $N$ flips.

$$p(\theta) = \text{Beta}(\theta|a_0, b_0) \qquad p(y|\theta) = \text{Bin}(y|N, \theta)$$

<span style="color: red;">

Where $\theta \in [0,1]$ is the unknown parameter and $y$ is observed number of successes, and $N$ is number of trials and $a_0, b_0 > 0$ are hyperparameters (pseudo-counts of prior successes and failures, respectively).

</span>

**Posterior** (conjugacy means the posterior is in the same family (like same function form like being the same kind of distributions) as the prior):
$$p(\theta|y) = \text{Beta}(\theta \mid y + a_0,\; N - y + b_0)$$

> **Why conjugacy works:** Multiply the Beta PDF by the Binomial PMF; the $\theta$-dependent terms combine to give another Beta.

**Posterior Predictive** (probability of $y^*$ heads in $N^*$ future flips):
$$p(y^*|y, N^*) = \int_0^1 \text{Bin}(y^*|N^*, \theta)\,\text{Beta}(\theta|a_N, b_N)\,d\theta = \text{BetaBin}(y^*|N^*, a_N, b_N)$$

**Posterior Mean (Bayes estimator):**
$$\hat{\theta}_{\text{Bayes}} = \mathbb{E}[\theta|y] = \frac{y + a_0}{N + a_0 + b_0}$$

Note: as $N \to \infty$, the posterior mean approaches the MLE $y/N$. The prior only matters when data is scarce.

The posterior mean is a **convex combination** of the prior mean $\theta_0 = \frac{a_0}{a_0 + b_0}$ and the MLE $\hat\theta_{\text{MLE}} = \frac{y}{N}$:

<span style="color: blue;">

$$\hat{\theta}_{\text{Bayes}} = (1-\lambda)\,\theta_0 + \lambda\,\hat{\theta}_{\text{MLE}}, \qquad \lambda = \frac{N}{N + a_0 + b_0}$$
 

</span>

 As $N \to \infty$, $\lambda \to 1$ and the prior is washed out.

##### **1.6 Maximum Likelihood Estimation (MLE)**




$$\hat{\theta}_{\text{MLE}} = \arg\max_\theta \log p(y|\theta)$$

For the Binomial:
$$\hat{\theta}_{\text{MLE}} = \frac{y}{N} = \frac{1}{N} \sum_{n=1}^N y_n$$

This is found by setting the derivative of the log-likelihood to zero and solving for $\theta$. The MLE is the parameter value that makes the observed data most probable under the model.


 Problem: if $y=0$, then $\hat{\theta}=0$ which is classic **overfitting** on small data.


The frequentist **95% confidence interval** (Wald interval) for $\theta^{\text{MLE}} = \hat\theta_{\text{MLE}}$ is:

 $$\text{CI} = \hat{\theta}_{\text{MLE}} \pm 1.96\sqrt{\frac{\hat{\theta}_{\text{MLE}}(1-\hat{\theta}_{\text{MLE}})}{N}}$$
> This relies on a Gaussian approximation and is **not** the same as a posterior credibility interval.

##### **1.7 Credibility intervals versus Confidence intervals**


A **95% credibility interval** $[\ell, u]$ satisfies:
$$P(\theta \in [\ell, u] | y) = 0.95$$
This is a direct probability statement about $\theta$ - much more intuitive than a frequentist confidence interval (there is $x$ probability that $\theta$ is in this interval).

Here for $95\%$ we usually use $1.96$ standard deviations because:

$$P(|Z| \leq 1.96) = 0.95 \quad \text{for } Z \sim \mathcal{N}(0,1)$$

For other values, like $90\%$ we find it via:
$$
P(|Z| \leq z) = 0.90 \quad \Rightarrow \quad z \approx 1.645
$$

##### **1.8 Glossary of probability operations**


> Evaluation means to evaluate the probability density of a random variable $X$ taking value $x$ for isntance $p(X=x)$

> Conditioning means deriving the distribution of $X$ conditioned on observed data $D$ so $p(X|D) = \frac{p(x,D)}{p(D)}$

> Maximization means finding the most likely value for a given distribution $\hat{x} = \argmax_x p(x|D)$

> Marginalizing is accounting for unobserved quantities like integrating out a latent variable $p(x) = \int p(x,y) dy$

> Sampling is to generate samples from a given probability distribution $x^{(i)} \sim p(x|D)$

> Computing expectations and moments is $\mu = \mathbb{E}[X] = \int x p(x) dx$

> Constructing invtervals is to find values $\ell$ and $u$ such that $p(\ell \leq x \leq u) = 0.95$ for instance

<a id='week2'></a>

<div class="alert alert-block alert-info">

### **Week 2 | predictors, plug-in & grid approximation**

Probabilistic machine learning, plug-in approximation, grid approximation, non-conjugate models, logistic regression

</div>

##### **2.0 Probabilistic machine learning**

A probabilistic model is specified by its joint distribution, and if we consider a model with two random variables $y$ (data) and $\theta$ (unknown parameters), then the joint distribution of all random variables can be expressed via product rule:
$$
p(\theta, y) = p(y|\theta)p(\theta)
$$
Where $p(y|\theta)$ is the likelihood and $p(\theta)$ is the prior.

The posterior distribution is obtained by conditioning on the observed data $y$:
$$
p(\theta|y) = \frac{p(y, \theta)}{p(y)} = \frac{p(y|\theta)p(\theta)}{\int p(y|\theta)p(\theta)d\theta}
$$
With $p(y)$ being the evidence or marginal likelihood.

The likelihood can be further decomposed using conditional independence for a broad class of models:

<span style="color: blue;">

$$
p(y| \theta) = p(y_1 | \theta)p(y_2 | \theta) \cdots p(y_N | \theta) = \prod_{n=1}^N p(y_n | \theta)
$$

</span>

The joint distribution is then:

<span style="color: blue;">

$$
p(\theta, y) = p(y| \theta)p(\theta) = \prod^N_{n=1} p(y_n | \theta) p(\theta)
$$

</span>

##### **2.1 The posterior predictive distribution**




The fully Bayesian prediction marginalizes over parameters:

<span style="color: blue;">

$$p(y^*|y) = \int p(y^*|\theta)\, p(\theta|y)\, d\theta = \mathbb{E}_{p(\theta|y)}[p(y^*|\theta)]$$

</span>

This **accounts for parameter uncertainty** - predictions are more conservative (wider uncertainty) than the plug-in. This is dervied by first formulating a joint distribution for all variables of interest like:

**Write the joint:**

$$
p(y^*, y, \theta) = p(y^*, y| \theta) p(\theta) = p(y^*| \theta)p(y| \theta)p(\theta)
$$
**Then condition on the observed data $y$:**
$$
p(y^*, \theta | y) = \frac{p(y^*, y, \theta)}{p(y)} = \frac{p(y^*| \theta)p(y| \theta)p(\theta)}{p(y)}
$$
T**hen we marginalized out $\theta$ using the sum rule** to get the posterior predictive distribution (this method is used many times in the course, so it is important to understand it well):
$$
p(y^* | y) = \int p(y^*, \theta | y) d\theta
$$

<span style="color: red;">

This results in the first expression above marked with blue. Here, $y^*$ is the new data point we want to predict, $y$ is observed data and $\theta$ is the unknown parameter we are marginalizing out.

</span>

(REMEMBER THAT THE SUM RULE IS FOR DISCRETE VARIABLES AND THE INTEGRAL IS FOR CONTINUOUS VARIABLES, SO IT DEPENDS ON THE TYPE OF VARIABLE WE ARE MARGINALIZING OUT (like Binomial is sumrule and Beta is integral))

##### **2.2 The plug-in (point-estimate) approximation**


Recall that the normal distribution over $x \in \mathbb{R}$ is given by:
$$
\mathcal{N} (x|\mu, \sigma^2) = \frac{1}{\sqrt{2\pi\sigma^2}}\exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right)
$$
With mean and variance as the two parameters. Furthermore, in limit $\sigma^2 \to 0$ then $x$ is constant $x=\mu$, and we say $x$ follows a Dirac's delta distribution centered at $\mu$:
$$
p(x) = \lim_{\sigma^2 \to 0} \mathcal{N}(x|\mu, \sigma^2) = \delta(x-\mu)
$$


If we assume $p(\theta|y) \approx \delta(\theta - \hat{\theta})$ (Dirac delta centred at a point estimate, so there is a single best parameter like MLE or MAP) then the sifting property of the delta distribution gives:

<span style="color: blue;">

$$p(y^*|y) = \int p(y^*|\theta) p(\theta|y) d\theta \approx \int p(y^*|\theta) \delta(\theta - \hat{\theta}) d\theta = p(y^*|\hat{\theta})$$

</span>

<span style="color: red;">

Where $\hat{\theta}$ is the point estimate (like MLE or MAP).

</span>

**Dirac delta properties:**
$$
\delta(x-\mu) = \begin{cases} \infty & x = \mu \\ 0 & x \neq \mu \end{cases}
$$
$$
\int \delta(x-\mu) dx = 1
$$
$$
\int f(x) \delta(x-\mu) dx = f(\mu)
$$
With the third equation being the sifting property and implies that $\mathbb{E}_{\delta(x-\mu)}[f(x)] = f(\mu)$ meaning that the expectation of $f(x)$ under the delta distribution is just $f(\mu)$.

> **Danger:** Plug-in ignores parameter uncertainty → **overconfident** predictions. This is what standard deep learning does!


##### **2.2.5 Non-conjugate models**

A model is **non-conjugate** when the posterior $p(\theta|y)$ does not belong to the same distributional family as the prior $p(\theta)$. This means the evidence $p(y) = \int p(y|\theta)p(\theta)d\theta$ is analytically intractable - but the intractable $p(y)$ is a *symptom*, not the cause.

**Why it happens:** The likelihood $p(y|\theta)$ has the "wrong shape" to combine cleanly with the prior. For example, a Gaussian prior on $w$ combined with a Bernoulli likelihood through a sigmoid $\sigma(w^Tx)$, so the sigmoid distorts things so the posterior is no longer Gaussian.

**Common examples:**

> Logistic regression: Gaussian prior + Bernoulli/sigmoid likelihood from Week 3

> GP classification: GP prior + Bernoulli likelihood from Week 6

> Poisson regression: Gaussian prior + Poisson likelihood through $e^{w^Tx}$ from Week 8

> Any neural network model from Week 12

**What we do about it:**

| Situation | Solution | Week |
|-----------|----------|------|
| Low-dimensional $\theta$ | Grid approximation | 2 |
| Unimodal posterior | Laplace approximation | 3 |
| General, needs samples | MCMC / Metropolis-Hastings | 8–9 |
| General, needs speed | Variational inference / BBVI | 10–11 |


**Key insight:** Non-conjugacy is the *motivation* for roughly half the course. Everything from Week 2 onwards is essentially asking "what do we do when conjugacy fails?"

##### **2.3 Grid approximation**


For non-conjugate models where the posterior is intractable (meaning we cannot write a 
closed-form expression for it, so instead of integrating we sum over a grid), 
discretise the parameter space.

**Algorithm:**
1. Define a grid of $M$ points: $\theta_1 < \theta_2 < \cdots < \theta_M$
2. Evaluate unnormalised posterior at each grid point:
$$\tilde{\pi}_i \propto p(\theta_i | y) \propto p(y|\theta_i)\,p(\theta_i)$$
3. Compute normalisation constant: $Z = \sum_{i=1}^M \tilde{\pi}_i$
4. Normalise to get grid weights: $\pi_i = \tilde{\pi}_i / Z$

<span style="color: red;">

Where $\theta_i$ are the grid points, $\tilde{\pi}_i$ are the unnormalised posterior values at those points, $Z$ is the normalisation constant and $\pi_i$ are the normalised weights that sum to 1.

This gives the grid approximation $q(\theta) = \sum_{i=1}^M \pi_i \delta(\theta - \theta_i)$,
a discrete distribution that places probability mass $\pi_i$ at each grid point $\theta_i$. ALSO, we evaluate unnormalised posterior at each grid point $\tilde{\pi}_i \propto p(\theta_i | y) \propto p(y|\theta_i)\,p(\theta_i)$ i.e. the likelihood times the prior so no need to compute normalisation constant $p(y)$

</span>

**Posterior summaries** are then just weighted sums:

> **Mean**: $\mathbb{E}[\theta] \approx \sum_i \theta_i \pi_i$

> **General expectation**: $\mathbb{E}[f(\theta)] \approx \sum_i f(\theta_i)\,\pi_i$

> **Posterior probability**: $p(\theta < c\,|\,y) \approx \sum_{i:\,\theta_i < c} \pi_i$

> **Posterior predictive**: $p(y^*|y) \approx \sum_i p(y^*|\theta_i)\,\pi_i$

**Limitation:** Curse of dimensionality - grid size grows as $M^D$ for $D$ parameters with $M$ points each,
so this is only practical for $D = 1$ or $D = 2$, also as the slides say "Grid approximations do not scale well beyond 3-4 dimensions". Furthermore, $M$ is balance between computaitonal cost and accuracy, and grid approximation is zero when evaluated outside the grid points, and gives often diminishing returns as $M$ increases.


##### **2.4 2D Grid approximation (exam style)**



Given a 2D grid with weights $\pi_{ij} = q(w_1^{(i)}, w_2^{(j)})$:

<span style="color: red;">

(where $\pi_{ij}$ is the normalized posterior weight at the grid point $(w_1^{(i)}, w_2^{(j)})$ computed as $\pi_{ij} \propto p(y|w_1^{(i)}, w_2^{(j)}) p(w_1^{(i)}, w_2^{(j)})$ and then normalised so that $\sum_{i,j} \pi_{ij} = 1$)

</span>

> **Marginal**:
$$q(w_1) = \sum_{j} \pi_{ij}$$

> **Mean**:
$$\mathbb{E}[w_1] = \sum_{i,j} w_1^{(i)} \cdot \pi_{ij}$$

> **Variance**:
$$\mathbb{V}[w_1] = \mathbb{E}[w_1^2] - (\mathbb{E}[w_1])^2, 
\quad \mathbb{E}[w_1^2] = \sum_{i,j} (w_1^{(i)})^2 \cdot \pi_{ij}$$

> **Posterior probability**:
$$p(w_1 < c\,|\,y) \approx \sum_{i:\,w_1^{(i)} < c}\sum_j \pi_{ij}$$

> **Predictive**:
$$p(y^*|y,x^*) \approx \sum_{i,j} p(y^*|w_1^{(i)},w_2^{(j)},x^*)\cdot \pi_{ij}$$

##### **2.4.1 Reading a pre-given grid table by hand - the 2025 exam pattern**


> The 2025 exam gave us a printed grid table (like Figure 1 below) and asked us to compute means, variances, and predictives **by hand**, with no code. This is the pattern.

**Setup:** We are given a grid table where cell $(i,j)$ contains $\pi_{ij} = q(w_1^{(i)}, w_2^{(j)})$,
with $\sum_{i,j} \pi_{ij} = 1$ (already normalized). Example from 2025 exam:

| $w_2 \backslash w_1$ | 0.5 | 0.6 | 0.7 |
|---|---|---|---|
| **0.3** | 0.09 | 0.12 | 0 |
| **0.2** | 0.12 | 0.20 | 0.12 |
| **0.1** | 0 | 0.12 | 0.23 |

(All other cells are 0. The non-zero weights sum to 1.00.)

**Posterior mean of $w_1$** - weight each $w_1$ value by the sum of its column:
$$\mathbb{E}[w_1] = \sum_{i,j} w_1^{(i)} \cdot \pi_{ij}
= 0.5(0.09+0.12+0) + 0.6(0.12+0.20+0.12) + 0.7(0+0.12+0.23)$$
$$= 0.5(0.21) + 0.6(0.44) + 0.7(0.35) = 0.105 + 0.264 + 0.245 = 0.614$$

**Posterior variance of $w_1$:**
$$\mathbb{E}[w_1^2] = \sum_{i,j} (w_1^{(i)})^2 \cdot \pi_{ij}
= 0.25(0.21) + 0.36(0.44) + 0.49(0.35) = 0.053 + 0.158 + 0.172 = 0.383$$
$$\mathbb{V}[w_1] = \mathbb{E}[w_1^2] - (\mathbb{E}[w_1])^2 = 0.383 - 0.614^2 = 0.383 - 0.377 = 0.006$$

**Posterior predictive $p(y^* \mid y, x^*)$** - for a non-linear model $f(x) = e^{w_1 + w_2 x}$
with Gaussian likelihood $p(y^* \mid w, x^*) = \mathcal{N}(y^* \mid e^{w_1 + w_2 x^*}, \sigma^2)$:

$$p(y^* \mid y, x^*) \approx \sum_{i,j} \mathcal{N}(y^* \mid e^{w_1^{(i)} + w_2^{(j)} x^*}, \sigma^2) \cdot \pi_{ij}$$

For $x^* = 4$, $y^* = 4$, $\sigma^2 = 0.5$ — compute $f_{ij} = e^{w_1^{(i)} + w_2^{(j)} \cdot 4}$ for
each non-zero cell, evaluate $\mathcal{N}(4 \mid f_{ij}, 0.5)$, multiply by $\pi_{ij}$, and sum.

```python
import jax.numpy as jnp
from jax.scipy.stats import norm

# Non-zero grid points from the table
w1_vals = jnp.array([0.5, 0.5, 0.6, 0.6, 0.6, 0.7, 0.7])
w2_vals = jnp.array([0.3, 0.2, 0.3, 0.2, 0.1, 0.2, 0.1])
pis    = jnp.array([0.09, 0.12, 0.12, 0.20, 0.12, 0.12, 0.23])

x_star, y_star, sigma2 = 4.0, 4.0, 0.5
f_vals = jnp.exp(w1_vals + w2_vals * x_star)          # predicted means
likelihoods = norm.pdf(y_star, loc=f_vals, scale=jnp.sqrt(sigma2))
p_pred = jnp.sum(likelihoods * pis)
```

**The 3-step by-hand checklist for any grid predictive:**
1. For each non-zero cell, compute the model output $f(x^* \mid w^{(i)}, w^{(j)})$
2. Evaluate the likelihood $p(y^* \mid f_{ij})$ at your target $y^*$
3. Multiply by $\pi_{ij}$ and sum - that is our answer

<span style="color: red;">

**Common mistake:** using the grid weights as if they are samples and averaging
$f_{ij}$ values instead of likelihood values. The predictive is a weighted average
of *likelihoods*, not of *function outputs*.

</span>

<a id='week3'></a>

<div class="alert alert-block alert-info">

### **Week 3 | bayesian linear regression**

Bayesian linear regression, posterior predictive distribution, hyperparameters

</div>

##### **3.0 Linear Regression**

If we have noise models:
$$
y_i = f(x_i|w) + \epsilon_i
$$
The non-linear feature extractors $\phi(\cdot)$ are called basis functions where:
$$
f(x|w) = \sum_{j=0}^M w_j \phi_j(x) = w^T\phi(x)
$$

<span style="color: red;">

Where $M$ is number of basis functions, $w_j$ is the weight for the $j$-th basis function and $\phi_j(x)$ is the $j$-th basis function evaluated at input $x$. The bias term is often included as $\phi_0(x) = 1$.

</span>


The probabilistic model of linear regression is when we have predictive distribution of $y^*$ given $x^*$ and data $D$ being our goal $p(y^*| D, x^*)$, then we model for the "signal":
$$
f(x_i | w) = w^T\phi(x_i)
$$
Here, the gaussian noise $\epsilon_i$ is assumed to be i.i.d so $y_i = f(x_i | w) + \epsilon_i$ with $\epsilon_i \sim \mathcal{N}(0, \sigma^2)$, so the **likelihood** for the $i$'th data point is:
$$
p(y_i| x_i, w, \sigma^2) = \mathcal{N}(y_i | w^T\phi(x_i), \sigma^2)
$$

<span style="color: red;">

Where $\sigma^2$ is the variance of the noise, and $w^T\phi(x_i)$ is the mean of the Gaussian distribution for $y_i$ given $x_i$ and parameters $w$.

</span>

This means given weights $w$ and input $x_i$, the output $y_i$ is gaussian distributed around mean $w^T\phi(x_i)$ with variance $\sigma^2$.

If we then use the maximum likelihood solution as a plug-in estimator then:
$$
p(y^*| D, x^*) = \mathcal{N}(y^* | \hat{w}_\text{MLE}^T \phi(x^*), \sigma^2)
$$

This is the plug-in approximation from section 2.2 applied to linear regression.

<span style="color: red;">

This is the plug-in predictive, $\hat{w}_\text{MLE}$ is treated as the real $w$,ignoring parameter uncertainty. The variance
$\sigma^2$ here only reflects observation noise, not uncertainty about $w$.

</span>

##### **3.0.5 Parameter estimation using maximum likelihood (linear models)**

The likelihood for a dataset $D = \{(x_i, y_i)\}_{i=1}^N$ is:
$$
p(y| w, \sigma^2) = \prod_{n=1}^N \mathcal{N}(y_n | f(x_n | w), \sigma^2)
$$
OR:

<span style="color: blue;">

$$
p(y| x, w, \sigma^2) = \prod_{n=1}^N \mathcal{N}(y_n | w^T\phi(x_n), \sigma^2) = \mathcal{N}(y| \phi w, \sigma^2 I)
$$

</span>

Where $\phi$ is the design matrix.

Then we take the logarithm and use the fact that $f(x_n | w) = w^T\phi(x_n)$ to get:
$$
\ln p(y| w, \sigma^2) = -\frac{N}{2}\ln(2\pi\sigma^2)-\frac{1}{2\sigma^2}\sum_{n=1}^N (y_n - w^T\phi(x_n))^2
$$
The MLE is then equivalent to minimizing the sum of squared errors:

<span style="color: blue;">

$$
\hat{w}_\text{MLE} = (\phi^T\phi)^{-1}\phi^Ty
$$
$$
\hat{\sigma}^2_\text{MLE} = \frac{1}{N}\sum_{n=1}^N (y_n - \hat{w}_\text{MLE}^T\phi(x_n))^2
$$

</span>

MLE IS ALWAYSS THE ABOVE BLUE FORMULA IF $p(y|\theta)$ (THE LIKELIHOOD) IS GAUSSIAN WITH MEAN BEING SOME $X$ MATRIX TIMES $\theta$, OR LIKE DESIGN MATRIX $\phi$ TIMES $w$ IN THIS CASE.


Finally, marginal likelihood in this setup is (also known as denominator in Bayes' theorem and independent of $w$):
$$
p(y)= \int p(y|w)p(w)dw = \mathbb{E}_{p(w)}[p(y|w)]
$$
With conjugate prior for the $w$:
$$
p(w)=\mathcal{N}(w|0, \alpha^{-1}I)
$$

##### **3.1 Generative versus discriminative classification**


| Approach | Model | Pros | Cons |
|----------|-------|------|------|
| **Generative** | Model $p(x\|y)$ and $p(y)$, then use Bayes to get $p(y\|x)$ | Can handle missing data, generate samples | Assumptions on $p(x\|y)$ may be wrong |
| **Discriminative** | Model $p(y\|x)$ directly | Often better calibrated; more flexible | Cannot handle missing inputs |

##### **3.1.5 Regularized Least Squares (RLS)**

If we add penalty term to maximum likelihood in order to prevent weights from becoming too large, we get:
$$
\tilde{E}_D(w)= \frac{1}{2}\sum_{n=1}^N (y_n -f(x_n | w))^2 + \frac{\lambda}{2} \|w\|^2
$$
This is **ridge regression**, or shrinkage or weight decay (has many names).

We can also write is as:
$$\hat{w}_{\text{MAP}} = \arg\max_w \log p(w|y) = (\alpha I + \beta\Phi^T\Phi)^{-1}\beta\Phi^T y = \arg\min_w \left\{\|y - \Phi w\|^2 + \frac{\alpha}{\beta}\|w\|^2\right\}$$

Ridge regularisation $\lambda = \alpha/\beta$ is a direct consequence of the Gaussian prior with precision $\alpha$.


##### **3.2 Logistic regression (discriminative)**


$$y_n | w, x_n \sim \text{Ber}(\sigma(w^T \phi(x_n)))$$

where $\sigma(a) = \frac{1}{1+e^{-a}}$ is the **logistic sigmoid**.

**Prior:**
$$w \sim \mathcal{N}(0, \alpha^{-1}I)$$

**Log-joint:**

<span style="color: blue;">

$$\log p(y, w) = \sum_n [y_n \log \sigma(f_n) + (1-y_n)\log(1-\sigma(f_n))] - \frac{\alpha}{2}w^T w + \text{const}$$

</span>

Here, $f_n = w^T \phi(x_n)$ is the linear predictor for the $n$-th data point.

The posterior $p(w|y)$ is **not Gaussian** (non-conjugate) — need approximate inference.


##### **3.3 The MAP estimator**

We have the maximum a posteriori (MAP) estimator:

<span style="color: blue;">

$$
\hat{w}_\text{MAP} = \argmax_w p(w|y) = \argmax_w \log p(w|y)
$$

</span>

With:
$$
p(w|y) \propto \mathcal{N}(y| \phi w, \sigma^2 I) \mathcal{N}(w|0, \alpha^{-1}I)
$$
Taking the logarithm gives:
$$
\ln p(w|y) \propto - \frac{\beta}{2} \sum_{n=1}^N (y_n - w^T \phi(x_n))^2 -\frac{\alpha}{2} \sum_{i=1}^D w_i^2 + \text{constant}
$$
The mode of the posterior (MAP) is equivalent to ridge regression with $\lambda = \frac{\alpha}{\beta}$ and to maximum likelihood when $\alpha \to 0$.

<span style="color: red;">

In practice, MAP means taking gradient of log posterior with respect to $w$ and setting it to zero, then solving for $w$, for linear regeression we get closed form solution $\hat{w}_\text{MAP} = (\alpha I + \beta\Phi^T\Phi)^{-1}\beta\Phi^T y$. But no closed form for logistic regression.

</span>

##### **3.3.5 Deriving the posterior distribution**

Now for deriving the posterior distribution of the weights we use the functional form of a generic $\mathcal{N}(w|m,S)$ which is:
$$
\ln \mathcal{N}(w | m, S) = -\frac{1}{2} w^T S^{-1} w + m^T S^{-1} w + K
$$
We match against the log-likelihood of the posterior, which we know from taking the 
log of the likelihood × prior:

$$
\ln p(w|y) \propto \ln p(y|w) + \ln p(w) 
= -\frac{\beta}{2}\|y - \Phi w\|^2 - \frac{\alpha}{2}w^Tw + K
= -\frac{1}{2}w^T(\beta\Phi^T\Phi + \alpha I)w + \beta y^T\Phi w + K
$$

Comparing with $\ln \mathcal{N}(w|m,S) = -\frac{1}{2}w^T S^{-1} w + m^T S^{-1} w + K$, so equating the coefficients we get first the **quadratic term**:

<span style="color: blue;">

$$
S^{-1} = \beta \phi^T \phi + \alpha I \Rightarrow S= (\beta \phi^T \phi + \alpha I)^{-1}
$$

</span>

**The linear term**

<span style="color: blue;">

$$
m^T S^{-1} = \beta y^T \phi \Rightarrow m = S \beta \phi^T y
$$

</span>

ADDITIONAL NOTES on model selection using marginal likelihood (evidence):

$$\log p(y|\alpha, \beta) = -\frac{N}{2}\log(2\pi) - \frac{1}{2}\log|\sigma^2 I + \alpha^{-1}\Phi\Phi^T| - \frac{1}{2}y^T(\sigma^2 I + \alpha^{-1}\Phi\Phi^T)^{-1}y$$

**Why does marginal likelihood select good models?**  

It balances data fit against model complexity. Overly complex models spread their prior probability too thinly → low marginal likelihood.


##### **3.4 Key equations of Bayesian linear regression (Bayesian linear regression)**

Given design matrix $\phi \in \mathbb{R}^{N \times D}$ and observations $y \in \mathbb{R}^N$:

<span style="color: blue;">

$$
p(w) = \mathcal{N}(w|0, \alpha^{-1}I) \quad \text{(prior)}
$$
$$
p(y|w) = \mathcal{N}(y|\phi w, \sigma^2 I) \quad \text{(likelihood)}
$$
$$
p(w|y) = \mathcal{N}(w|m, S) \quad  \text{(posterior)}
$$
$$
p(y) = \mathcal{N}(y|0, \sigma^2 I + \alpha^{-1} \phi \phi^T) \quad \text{(marginal likelihood)}
$$

</span>

With:

<span style="color: blue;">

$$
m = \beta S \phi^T y

$$
$$
S = (\alpha I + \beta \phi^T \phi)^{-1}
$$

</span>

<span style="color: red;">

Here $\alpha = \frac{1}{\tau^2}$ and $\alpha^{-1}$ is prior variance of weights (larger $\alpha$ means stronger regularisation), and $\beta = \frac{1}{\sigma^2}$ is the precision of the noise (smaller $\sigma^2$ means less noise, so higher precision). $S$ is the posterior covariance of the weights, and $m$ is the posterior mean of the weights.


</span>

##### **3.5 Linear Gaussian systems**

For linear systems, the Gaussian distribution is conjugate to itself, and the posterior of a linear Gaussian model with Gaussian prior is ALSO Gaussian:

<span style="color: blue;">

$$
p(y|z) = \mathcal{N}(y|Wz + b, \Sigma_y) \quad \text{(likelihood)}
$$
$$
p(z) = \mathcal{N}(z | \mu_z, \Sigma_z) \quad \text{(prior)}
$$

</span>

The joint distribution $p(y,z)$ is:
<span style="color: blue;">

$$
p(z,y) = \mathcal{N}\left( \begin{bmatrix} z \\ y \end{bmatrix} \Bigg| \begin{bmatrix} \mu_z \\ W\mu_z + b \end{bmatrix}, \begin{bmatrix} \Sigma_z & \Sigma_z W^T \\ W\Sigma_z & W\Sigma_z W^T + \Sigma_y \end{bmatrix} \right)
$$

</span>

The posterior distribution of $z$ given $y$ is:

<span style="color: blue;">

$$
p(z|y) = \mathcal{N}(z|\mu_{z|y}, \Sigma_{z|y})
$$
$$
\Sigma^{-1}_{z|y} = \Sigma^{-1}_z + W^T \Sigma^{-1}_y W
$$
$$
\mu_{z|y} = \Sigma_{z|y} [W^T\Sigma_y^{-1}(y - b) + \Sigma_z^{-1} \mu_z]
$$

</span>

The marginal distribution of $y$ is:

<span style="color: blue;">

$$
p(y) =  \int p(y|z)p(z)dz = \mathcal{N}(y|W\mu_z + b, \Sigma_y+W\Sigma_z W^T)
$$

</span>


##### **3.6 Posterior predictive for linear regression**


<span style="color: red;">

Now this is about MAKING predictions, say we defined a linear regression model $y_i = w^T\phi(x_i)+ \epsilon_i$ where we dervied the posterior distribution $p(w|y) = \mathcal{N}(w|m,S)$, now the next goal is making predictions for the new $x^*$:

</span>

$$
p(y^*|x^*,w ) = \mathcal{N}(y^*|w^T \phi(x^*), \sigma^2)
$$


The posterior predictive distribution is then:

<span style="color: blue;">

$$
p(y^*|y, x^*) = \int p(y^*| x^*, w)p(w|y)dw = \int \mathcal{N}(y^*|w^T \phi_*, \sigma^2) \mathcal{N}(w|m,S) dw = \mathcal{N}(y^*| m^T \phi_*, \sigma^2 + \phi_*^T S \phi_*)
$$

</span>

(note from this above, we still don't know $w$, which is why we need section 3.4 formulas first)

Furthermore, the corresponding posterior distribution is:
$$
p(f^* | y, x^*) = \mathcal{N}(y^*| m^T \phi_*, \phi_*^T S \phi_*)
$$
(this is also shown further below)

##### **3.6.5 Posterior predictive for classification (Laplace)**


With Laplace approx $p(w|y) \approx \mathcal{N}(w|\hat{w}, S)$ we get intractable integral.

This is **analytically intractable**. Three strategies:
1. **Sampling:** Draw $w^{(s)} \sim q(w)$, estimate $p(y^*=1) \approx \frac{1}{S}\sum_s \sigma(w^{(s)T}\phi^*)$
2. **Numerical integration**
3. **Probit approximation** (see below)


For **classification** (logistic regression), the likelihood is Bernoulli with a sigmoid, 
so the posterior predictive requires integrating a sigmoid against a Gaussian:

$$p(y^*=1|y,x^*) = \int \sigma(w^T\phi(x^*))\,\mathcal{N}(w|\hat{w},S)\,dw \quad \text{intractable integral}$$

This is analytically intractable. The **probit approximation** approximates the 
sigmoid $\sigma(\cdot)$ with the Gaussian CDF $\Phi(\cdot)$, making the integral tractable.

Approximate the sigmoid with the Gaussian CDF:
$$\sigma(a) \approx \Phi\!\left(\frac{a}{\sqrt{\pi/8}}\right)$$

Then the integral becomes tractable:

<span style="color: blue;">

$$p(y^*=1|y,x^*) \approx \Phi\!\left(\frac{\mu_{f^*}}{\sqrt{\frac{8}{\pi} + \sigma^2_{f^*}}}\right)$$

</span>

<span style="color: red;">

Where $\mu_{f^*} = \hat{w}^T\phi^*$ and $\sigma^2_{f^*} = (\phi^*)^T S \phi^*$, AND we use $\frac{8}{\pi}$ BECAUSE the professor uses that in his exam solutions.

</span>

> **Key property of probit approx:** The posterior predictive mean of $f^*$ is scaled down by the variance — uncertainty flattens the sigmoid towards 0.5.

> **Notation:** $\Phi(\ldots)$ is a scalar (a probability), not a distribution.

> If LHS is $p(y^*=1|y,x^*)$, just set it equal to $\Phi(\ldots)$ directly.

> If LHS is $p(y^*|y,x^*)$ (no $=1$), must wrap: $p(y^*|y,x^*) = \text{Ber}(y^*|\Phi(\ldots))$.

> Both here say the same thing! $\text{Ber}(y^*|p^*)$ just means $p(y^*=1)=p^*$ and $p(y^*=0)=1-p^*$.

> Note that unless $\phi$ is used at feature mappings like design matrix $\phi = [1 \: x \: x^2 \: \cdots]$ then usually $\phi(x)=\phi=x$ and likewise for $\phi^*$.

**Python version (both lines are the same formula, just written differently):**
```python
from scipy.stats import norm
# Posterior predictive
p1 = norm.cdf(mu_f / jnp.sqrt(8/jnp.pi + var_f))
# Prior predictive — plug in mu_f=0, var_f=k(x*,x*)
p1_prior = norm.cdf(0 / jnp.sqrt(8/jnp.pi + k_star_star))  # always 0.5
```

##### **3.7 The evidence approximation**


With a flat prior $p(\alpha,\beta) \propto 1$, maximising the posterior over $\alpha, \beta$ 
reduces to maximising the marginal likelihood (**type-II MLE** / evidence approximation):

<span style="color: blue;">

$$\hat{\alpha}, \hat{\beta} = \arg\max_{\alpha,\beta}\, p(y|\alpha,\beta)$$

</span>

<a id='week4'></a>

<div class="alert alert-block alert-info">

### **Week 4 | bayesian classification & laplace approximations**

Bayesian versus classical datasets, bayesian methods for classification, bayesian logistic regression, laplace approximations, the posterior predictive distribution

</div>

##### **4.0 The model**


$$y_n = \underbrace{w^T \phi(x_n)}_{f(x_n)} + \epsilon_n, \quad \epsilon_n \sim \mathcal{N}(0, \sigma^2)$$

With prior $p(w) = \mathcal{N}(w|0, \alpha^{-1}I)$ and $\beta = 1/\sigma^2$.

| Distribution | Expression |
|---|---|
| **Prior** | $p(w) = \mathcal{N}(w\|0, \alpha^{-1}I)$ |
| **Likelihood** | $p(y\|w) = \mathcal{N}(y\|\Phi w, \sigma^2 I)$ |
| **Posterior** | $p(w\|y) = \mathcal{N}(w\|m, S)$ |
| **Marginal likelihood** | $p(y) = \mathcal{N}(y\|0, \sigma^2 I + \alpha^{-1}\Phi\Phi^T)$ |

**Posterior parameters (must memorise!):**

<span style="color: blue;">


$$S = (\alpha I + \beta \Phi^T\Phi)^{-1}, \qquad m = \beta S \Phi^T y$$

</span>


 > See section 3.4 for key equations where $\Phi \in \mathbb{R}^{N \times D}$ is design matrix with row $n$ equal to $\phi(x_n)^T$.

##### **4.1 Predictive distributions**

For a new point $x^*$ with $\phi^* = \phi(x^*)$ (like if they say $f^* = w_1 + w_2x^*$ then $\phi^* = [1, x^*]^T$), so the general predictive distribution is:

<span style="color: blue;">

$$p(f^*|y, x^*) = \mathcal{N}(f^* | \underbrace{m^T\phi^*}_{\text{pred. mean}},\; \underbrace{(\phi^*)^T S \phi^*}_{\text{epistemic var.}})$$

$$p(y^*|y, x^*) = \mathcal{N}(y^* | m^T\phi^*,\; (\phi^*)^T S \phi^* + \underbrace{\sigma^2}_{\text{aleatoric}})$$

</span>

<span style="color: red;">

Where $\phi^* = \phi(x^*)$ is the feature vector for the new input $x^*$, $m^T\phi^*$ is the predictive mean, $(\phi^*)^T S \phi^*$ is the epistemic variance (uncertainty about the function due to limited data) and $\sigma^2$ is the aleatoric variance (inherent noise in the data). $S$ is posterior covariance and $m$ is posterior mean of the weights.

</span>

> **Why does variance decompose this way?**  
> $y^* = f^* + \epsilon^*$ and $f^*$, $\epsilon^*$ are independent, so $\mathbb{V}[y^*] = \mathbb{V}[f^*] + \mathbb{V}[\epsilon^*]$.

> **IMPORTANT: $p(f^*|y,x^*)$ is ALWAYS Gaussian regardless of model.**

The second formula for $p(y^*|y,x^*)$ depends on the likelihood:
 
| Likelihood | $y^*$ | $p(y^*\|y,x^*)$ |
|---|---|---|
| Gaussian (regression) | $\mathbb{R}$ | $\mathcal{N}(y^*\|m^T\phi^*, (\phi^*)^TS\phi^* + \sigma^2)$ ← blue formula |
| Bernoulli (classification) | $\{0,1\}$ | $\text{Ber}(y^*\|p^*)$ where $p^* = \Phi\!\left(\frac{\mu_*}{\sqrt{8/\pi + \sigma^2_*}}\right)$ |
| Poisson (count) | $\{0,1,2,...\}$ | Use Monte Carlo sampling |


##### **4.2 Generative modeling**

Say we have binary classification $y_n \in \{0,1\}$, then we can model the likelihood as:
$$
p(y_n = 1 | x_n) = \frac{p(x_n|y_n=1)p(y_n=1)}{p(x_n)}
$$
Where, $p(x_n | y_n)$ is the class-conditional distribution, $p(y_n=k)=\pi_k$ is the prior porbabilities, and $p(x_n)$ is marginal data density, which is a mixture distirbution and obtained using the sum rule:

<span style="color: blue;">

$$
p(x_n) = \sum_{k\in \{0,1\}} p(x_n|y_n=k)p(y_n=k) = \pi_0 p(x_n|y_n=0) + \pi_1 p(x_n|y_n=1)
$$

</span>
When plugging this back in, we get:

$$
p(y_n = 1 | x_n) = \frac{\pi_1 p(x_n|y_n=1)}{\pi_0 p(x_n|y_n=0) + \pi_1 p(x_n|y_n=1)}
$$

We devide by the numerator and define:
$$
a= \ln \frac{\pi_1 p(x_n|y_n=1)}{\pi_0 p(x_n|y_n=0)}
$$
Then we get the logistic sigmoid (with the inverse being the logit function):

<span style="color: blue;">

$$
\sigma(a) = \frac{1}{1+e^{-a}}  = p(y_n=1|x_n)
$$
$$
a = \ln \frac{\sigma(a)}{1-\sigma(a)} = \text{logit}(\sigma(a))
$$

</span>


##### **4.3 Generative modelling for multi-class problems**

If we know have $K$ classes and:
$$
a_k = \ln p(x_n | y_n=k) p(y_n=k)
$$
Then for $K$ classes we have:

<span style="color: blue;">

$$
p(y_n = k | x_n) = \frac{e^{a_k}}{\sum_{j=1}^K e^{a_j}} = \text{softmax}(a)_k
$$

</span>

##### **4.4 Discriminative modeling**

For discriminative modelling, we directly assume a functional form for $a(x)$ an example with logistic regression is:
$$
p(y_n = 1 | x_n) = \frac{1}{1 + e^{-a}} = \sigma(\phi(x_n)^T w)
$$
Where we model each observation with a Bernoulli distribution with probability $\sigma(\phi(x_n)^T w)$, and then we estimate the parameters $w$ using MAP for bayesian inference with the likelihood function:
$$
p(y|w,X) = \prod_{n=1}^N \sigma (\phi(x_n)^T w)^{y_n} (1 - \sigma(\phi(x_n)^T w))^{1-y_n}
$$

##### **4.5 Bayesian logistic regression/ classification**


We place a Gaussian prior on the weights:
$$p(w) = \mathcal{N}(w|0, \alpha^{-1}I)$$
The posterior $p(w|y) \propto p(y|w)p(w)$ is **non-Gaussian** (sigmoid likelihood 
breaks conjugacy) → use Laplace approximation (section 4.6).

**Log-joint** (needed for MAP and Laplace):
$$\log p(y,w) = \sum_n [y_n \log \sigma(f_n) + (1-y_n)\log(1-\sigma(f_n))] 
- \frac{\alpha}{2}w^Tw + \text{const}$$

Where $f_n = w^T\phi(x_n)$ is the linear predictor.

**Gradient** (needed for MAP via gradient ascent):
$$\nabla_w \log p(y,w) = \Phi^T(y - \sigma(f)) - \alpha w$$

where $\sigma(f) = [\sigma(f_1), \ldots, \sigma(f_N)]^T$ is the vector of predicted probabilities.

**Hessian** (needed for Laplace covariance $S = (-H)^{-1}$) is shown in 4.6.

##### **4.6 Laplace approximation**

**Idea:** Approximate the posterior with a Gaussian centred at the MAP estimate, so, the Laplace approximation is a method for approximating intractable probability densities, and is defined:
$$
q(w) = \mathcal{N}(w | w_\text{MAP}, A^{-1})
$$

**Steps:**

<span style="color: blue;">

1. Find MAP: $\hat{w} = \arg\max_w \log p(y|w) + \log p(w)$  (use gradient ascent / scipy.minimize), this is where we locate the mode of $p(w|y)$.

2. Compute Hessian: $H = \nabla^2 \log p(y,w)|_{w=\hat{w}}$

3. Approximate: $p(w|y) \approx q(w) = \mathcal{N}(w | \hat{w},\; S)$ where $S = (-H)^{-1}$

    Note: $A = -H$ is the negative Hessian, so $A^{-1} = S = (-H)^{-1}$. Both notations appear in the slides.

4. Use $q(w)$ for predictions, e.g. $p(y^*|y,x^*) \approx \int p(y^*|x^*,w) q(w) dw$ (use Monte Carlo sampling or probit approximation)

</span>


**Why does this work?** It works because a second-order Taylor expansion of $\log p(y,w)$ around the mode gives a quadratic (which corresponds to a Gaussian). So $-H$ must be positive definite at a maximum.

**For logistic regression:**

$$H = -\Phi^T W \Phi - \alpha I$$

<span style="color: red;">

Where $f_n = \phi(x_n) w$ is the linear predictor for the $n$-th data point, $\sigma(f_n)(1-\sigma(f_n))$ is the variance of the Bernoulli distribution with probability $\sigma(f_n)$, and $\Phi$ is the design matrix, and $W = \text{diag}(\sigma(f_n)(1-\sigma(f_n)))$ is the diagonal matrix of second-derivative weights.

</span>

##### **4.6.5 Laplace approximation for the marginal likelihood**

First a second order Taylor approximation for $\ln f(w)$, where we STILL denote $A$ as the Hessian:
$$
\ln f(w) \approx \ln f(w_\text{MAP}) -\frac{1}{2}(w - w_\text{MAP})^T A (w - w_\text{MAP})
$$
By assumption then the posterior is proportional to $f(w)$, so we can write:
$$
p(w|y) = \frac{1}{Z} f(w) \Rightarrow Z = \int f(w) dw
$$
Plugging in the approximation for $\ln f(w)$ gives:
$$
Z = f(w_\text{MAP}) \frac{(2\pi)^{D/2}}{\sqrt{|A|}}
$$
Using $f(w)=p(y|w)p(w)$ gives the Laplace approximation for the marginal likelihood:

<span style="color: blue;">

$$
\ln p(y) \approx \ln p(y|w_\text{MAP}) + \ln p(w_\text{MAP}) + \frac{D}{2}\ln(2\pi) - \frac{1}{2}\ln |A|
$$

</span>

<span style="color: red;">

Where $w_\text{MAP}$ is the MAP estimate, $A = - \nabla\nabla \ln p(y|w)p(w)|_{w=w_\text{MAP}}$ is the negative Hessian of the log-joint at the MAP (positive definite at maximum), and $D$ is the dimensionality of $w$. This is used for **model selection** like to compare $\ln p(y)$ across different models (e.g. different basis functions) and select the one with highest marginal likelihood.

</span>

<a id='week5'></a>

<div class="alert alert-block alert-info">

### **Week 5 | Gaussian process regression**

Prior distribution for function spaces, gaussian process regression, covariance functions, hyperparameters, marginal likelihood

</div>

##### **5.1 From parameters to functions**

Bayesian linear model: $f(x) = w^T\phi(x)$, put a prior on $w$. 

**Gaussian Process** is when we put a prior **directly on the function** $f$, with the formal defintion being a Gaussian process (GP) is a collection of random variables, any finite number of which have a joint Gaussian distribution.

<span style="color: blue;">


$$f(x) \sim \mathcal{GP}(m(x),\; k(x,x'))$$

</span>

A GP is completely specified by:

> **Mean function:** $m(x) = \mathbb{E}[f(x)]$ (usually $m(x) = 0$)

> **Covariance function (kernel):** $k(x,x') = \text{Cov}[f(x), f(x')] = \mathbb{E}[(f(x)-m(x))(f(x')-m(x'))]$

> The above expressions mean that $f(x)$ and $f(x')$ are jointly Gaussian distributed with covaraicen $k(x,x')$.

Here, any finite collection of function values is jointly Gaussian:
$$[f(x_1), \dots, f(x_N)]^T \sim \mathcal{N}(\mathbf{0}, K) \quad\text{where } K_{ij} = k(x_i, x_j)$$
Here $K= \frac{1}{\alpha} \Phi \Phi^T$ is the covariance matrix of the function values at the training points, and:

<span style="color: blue;">

$$
K_{ij} = \text{cov}(y_i, y_j) = \frac{1}{\alpha}\phi(x_i)^T \phi(x_j) = k(x_i, x_j)
$$

</span>

<span style="color: red;">

Here $K$ is the connection between GP and Bayesian linear regression, where we place a prior $p(w) = \mathcal{N}(0, \alpha^{-1}I)$ on weights which induces a GP prior on functions with kernel $k(x_i, x_j) = \frac{1}{\alpha}\phi(x_i)^T \phi(x_j)$.

</span>

##### **5.2 Common kernel functions**


| Kernel | Formula | Stationary? | Isotropic? | Smoothness |
|--------|---------|-------------|------------|------------|
| **Squared Exponential** (RBF) | $\kappa^2 \exp\!\left(-\frac{\|x-x'\|^2}{2\ell^2}\right)$ | yes | yes | Infinitely differentiable |
| **Matérn 1/2** | $\kappa^2 \exp\!\left(-\frac{\|x-x'\|}{\ell}\right)$ | yes | yes | Continuous, not differentiable |
| **Matérn 3/2** | $\kappa^2(1+\frac{\sqrt{3}\|x-x'\|}{\ell})\exp(\cdots)$ | yes | yes | Once differentiable |
| **Linear** | $\alpha^{-1}x^Tx'$ | no | no | - |
| **ARD** | $\kappa^2\exp(-\frac{1}{2}(x-x')^T L^{-1}(x-x'))$ | yes | no | Inf. differentiable |

**Hyperparameters:**

> $\kappa > 0$: **magnitude** which controls the vertical scale of functions

> $\ell > 0$: **lengthscale** which controls how quickly functions vary (small $\ell$ = wiggly, large $\ell$ = smooth)

<span style="color: blue;">

**Stationary:**

$k(x,x')$ depends only on $x - x'$. 

**Isotropic:**

$k(x,x')$ depends only on $\|x - x'\|$.

</span>


##### **5.3 GP regression**

Model:
$$y_n = f(x_n) + \epsilon_n$$
$$\epsilon_n \sim \mathcal{N}(0,\sigma^2)$$

Likelihood for all datapoints assuming homoscedastic noise (so $\sigma^2$ is the same for all $n$):

$$
p(y|f) = \prod_{n=1}^N \mathcal{N}(y_n | f_n, \sigma^2) = \mathcal{N}(y|f, \sigma^2 I)
$$

Prior predictive:
$$p(y^*|x^*) = \mathcal{N}\!\left(y^* | 0,\; k(x^*,x^*) + \sigma^2\right)$$


> **Why this likelihood?** The model says $y_n = f_n + \epsilon_n$ with $\epsilon_n \sim \mathcal{N}(0,\sigma^2)$. Conditioned on $f_n$, $y_n$ is just $f_n$ plus Gaussian noise — so you read the likelihood directly off the model definition. Independence across $n$ gives the product form, which collapses to a multivariate Gaussian with diagonal covariance $\sigma^2 I$.

**Joint distribution:**

<span style="color: blue;">

$$p(y, y^*) = \mathcal{N}\!\left(\begin{bmatrix}y\\y^*\end{bmatrix}\bigg| 0,\; \begin{bmatrix}K + \sigma^2 I & k_*\\ k_*^T & c\end{bmatrix}\right)$$

</span>

<span style="color: red;">

Where:

> $K + \sigma^2 I$ is covariance of noisy training observations $y$ with $K = [k(x_n, x_m)]_{n,m=1}^N$. GP prior covariance $K$ plus observation noise $\sigma^2 I$. 

> $k_* = [k(x^*, x_n)]_{n=1}^N$ is cross-covariance between the **test point** $x^*$ and the **training points**. No noise here because we are correlating the latent function values, not noisy observations.

> So $k_*$ is a vector of shape $(N,)$ where the $n$-th element is $k(x^*, x_n)$ - the kernel evaluated between test and $n$'th training point.

> $c = k(x^*,x^*) + \sigma^2$ is variance of the **noisy test observation** $y^*$, prior variance plus noise.


</span>

**The two predictive distributions, and when to use which:**

| | $p(f^*\|y, x^*)$ | $p(y^*\|y, x^*)$ |
|---|---|---|
| **What it is** | Posterior over latent function | Posterior over noisy observation |
| **Variance** | $k(x^*,x^*) - k_*(K+\sigma^2 I)^{-1}k_*^T$ | same $+ \sigma^2$ |
| **When to use** | GP classification, epistemic uncertainty only | Predicting actual new observations |
| **Slides notation** | Not shown separately | $p(y^*\|y)$ with $c = k(x^*,x^*)+\sigma^2$ |

**Slides vs notes:** The slides compute $p(y^*|y)$ directly by absorbing $\sigma^2$ into $c = k(x^*,x^*)+\sigma^2$. The notes split it into two steps ($p(f^*|y)$ then add $\sigma^2$). **They give identical results** so the slides just do it in one shot.

**Posterior predictive for $f^*$** (latent function, no observation noise):

<span style="color: blue;">

$$p(f^*|y, x^*) = \mathcal{N}(f^* | \mu_{f^*|y},\; \sigma^2_{f^*|y})$$

$$\mu_{f^*|y} = k_*(K + \sigma^2 I)^{-1} y$$
$$\sigma^2_{f^*|y} = k(x^*,x^*) - k_*(K + \sigma^2 I)^{-1} k_*^T$$

</span>

**Posterior predictive for $y^*$** (observation, includes noise):

<span style="color: blue;">

$$\boxed{p(y^*|y, x^*) = \mathcal{N}(y^* | \mu_{y^*|y},\; \sigma^2_{y^*|y})}$$

$$\mu_{y^*|y} = \mu_{f^*|y} = k_*(K + \sigma^2 I)^{-1} y$$
$$\sigma^2_{y^*|y} = \sigma^2_{f^*|y} + \sigma^2 = k(x^*,x^*) - k_*(K + \sigma^2 I)^{-1} k_*^T + \sigma^2$$

</span>

(THESE FORMULAS ARE FOR GAUSSIAN PROCESSREGRESSION (GP REGRESSION)!!!)

> **Zero-mean prior assumption:** Both formulas assume $m(x) = 0$. If the prior mean is non-zero, replace $y$ with $y - \mu_X$ in the mean formula.

> **Exam rule of thumb:** If the question says $y_n = f(x_n) + \epsilon_n$ and asks for $p(y^*|y,x^*)$, use the $y^*$ formulas (add $\sigma^2$ to variance). If it asks for $p(f^*|y,x^*)$, use the $f^*$ formulas (no extra $\sigma^2$).

##### **5.3.5 Covariance for GP regression**


The question says **"prior covariance"** so $f(x_n)$ being unobserved is completely irrelevant. We never need to "know" $f(x_n)$ because the GP prior tells you the covariance structure through the kernel alone.

To summarize:

| Question asks for | Formula | Need $y$? |
|---|---|---|
| Prior covariance $\text{Cov}(f, f^*)$ | $k(x_n, x^*)$ | NO |
| Prior variance $\text{Var}(f^*)$ | $k(x^*, x^*)$ | NO |
| Posterior mean $\mu_{f^*\|y}$ | $k_*(K+\sigma^2I)^{-1}y$ | YES |
| Posterior variance $\sigma^2_{f^*\|y}$ | $k(x^*,x^*) - k_*(K+\sigma^2I)^{-1}k_*^T$ | YES |

Whenever we see the word **"prior"** it is kernel only, plug and chug. The mention of $f(x_n)$ is indeed bait to make you think you need to recover the latent function values. You don't.

##### **5.4 Gaussian conditioning formula (general)**


For jointly Gaussian $(y_1, y_2)$ with mean $\mu$ and covariance $\Sigma$ so $\mathcal{N}(y|\mu, \Sigma)$ then we have:

<span style="color: blue;">

$$p(y_1|y_2) = \mathcal{N}(y_1 | \mu_{1|2}, \Sigma_{1|2})$$
$$\mu_{1|2} = \mu_1 + \Sigma_{12}\Sigma_{22}^{-1}(y_2 - \mu_2)$$
$$\Sigma_{1|2} = \Sigma_{11} - \Sigma_{12}\Sigma_{22}^{-1}\Sigma_{21} = \Lambda_{11}^{-1}$$

</span>

Where:
$$
\mu = \begin{bmatrix} \mu_1 \\ \mu_2 \end{bmatrix}, \quad
\Sigma = \begin{bmatrix} \Sigma_{11} & \Sigma_{12} \\ \Sigma_{21} & \Sigma_{22} \end{bmatrix}
$$
The precision matrix $\Lambda = \Sigma^{-1}$ can be partitioned as:
$$\Lambda = \begin{bmatrix} \Lambda_{11} & \Lambda_{12} \\ \Lambda_{21} & \Lambda_{22} \end{bmatrix}$$

<span style="color: red;">

GP regression posterior is derived by applying this formula to the joint $p(y, f^*)$ - so identify $y_1 = f^*$, $y_2 = y$ and read off $\Sigma_{12} = k_*^T$, $\Sigma_{22} = K + \sigma^2 I$, $\Sigma_{11} = k(x^*, x^*)$ to get the posterior mean and variance formulas.

</span>

##### **5.5 Marginal likelihood for hyperparameter selection**


If we let $\theta$ denote all hyperparameters, then the marginal likelihood for Gaussian likelihood:

<span style="color: blue;">

$$
p(y|\theta) = \int p(y|f)p(f|\theta_k) df = \int \mathcal{N}(y|f, \beta^{-1} I)\mathcal{N}(f|0,K) df = \mathcal{N}(y|0, \beta^{-1} I + K)
$$

</span>

The hyperparameters of the model can be tuned by optimizing the marginal likelihood, where we in practive compute the gradient of $p(y|\theta)$ with respect to $\theta$ and use numerical optimization. The gradients of the marginal likelihood wrt. hyperparameters are:
$$
\frac{\partial}{\partial \theta_j} \log p(y| \theta) = \frac{1}{2} \text{tr} \left( (\alpha \alpha^T - (K+\sigma^2 I)^{-1}) \frac{\partial K}{\partial \theta_j} \right)
$$

<span style="color: red;">

Where $\alpha = (K+\sigma^2 I)^{-1} y$ and $\frac{\partial K}{\partial \theta_j}$ depends on the specific choice of kernel (this is NOT the same as prior precision used in weeks 3-4).

</span>

Then we have:

<span style="color: blue;">

$$\log p(y|\theta) = -\frac{N}{2}\log(2\pi) - \frac{1}{2}\log|K+\sigma^2 I| - \frac{1}{2}y^T(K+\sigma^2 I)^{-1}y$$

</span>

Optimize this numerically with respect to $\theta = \{\kappa, \ell, \sigma\}$ so much more efficient than cross-validation.

> Remember: $\log|K+\sigma^2 I|$ is the log of the determinant, not the determinant of $\log$.

We can find more about $K$ in week 5 notebook, but it states that it is defined as:

$$
\begin{align*}
\mathbf{K}_{nm}  = k(\mathbf{x}_n, \mathbf{x}_m) + \epsilon \delta(\mathbf{x}_n-\mathbf{x}_m),
\end{align*}
$$

<span style="color: red;">

Where $\delta(x_n-x_m)$ is $1$ if $x_n=x_m$ and $0$ otherwise (kronecker delta) and $\epsilon$ is a small number to ensure numerical stability (jitter) to $K$ is positive definite. 

</span>

##### **5.6 Computational considerations**


> Matrix inversion $C^{-1}$ so $(K + \sigma^2 I)^{-1}$ has $\mathcal{O}(N^3)$, so a cubic cost if $C \in \mathbb{R}^{N \times N}$.

> Memory is $\mathcal{O}(N^2)$.

> If $A \in \mathbb{R}^{N \times N}$ and $b \in \mathbb{R}^M$ then the cost of computing $Ab$ is $\mathcal{O}(N M)$.

> **Cholesky decomposition** is used in practice to avoid direct inversion and compute log-determinants stably.

> In practice, solve $(K+\sigma^2 I )\alpha = y$ via Cholesky rather than explicitly computing $(K+\sigma^2 I)^{-1}$.

**How Cholesky works in practice:**
1. Decompose $K + \sigma^2 I = LL^T$ where $L$ is lower triangular
2. Solve $L\alpha' = y$ (forward substitution), then $L^T\alpha = \alpha'$ (back substitution) → gives $\alpha = (K+\sigma^2I)^{-1}y$
3. Log-determinant: $\log|K+\sigma^2 I| = 2\sum_i \log L_{ii}$ (sum of log diagonal entries of $L$)

Cost is still $\mathcal{O}(N^3)$ for the decomposition, but more numerically stable than direct inversion.

<a id='week6'></a>

<div class="alert alert-block alert-info">

### **Week 6 | GP classification & advanced kernels**

Covariance functions, gaussian processes in practice, gaussian process classification, neural networks for probabilistic modelling
</div>

##### **6.1 Gaussian process (GP) classification model**


Three equivalent ways to do binary classification:

| Model | How $f$ is defined |
|---|---|
| Bayesian logistic regression | $f(x_n) = \phi(x_n)^T w$, $w \sim \mathcal{N}(0, \alpha^{-1}I)$ |
| **GP classification** | $f \sim \mathcal{GP}(0, k(x,x'))$ |
| Neural network | $f = \text{NN}(x_n \| w)$ |

All three use the same likelihood:
$$y_n \sim \text{Ber}(\sigma(f(x_n))), \quad \sigma(a) = \frac{1}{1+e^{-a}}$$

**Key problem:** unlike regression, $p(y|f)$ is Bernoulli (non-Gaussian), so $p(f|y)$ is **intractable**. Solution: Laplace approximation.


##### **6.2 GP Classification pipeline (3 steps), prior is now on $f$ and NOT $w$**


**STEP 1 use Laplace approximation to find $q(f) \approx p(f|y)$**

Maximise the log-joint to find MAP estimate $\hat{f}$:
$$\log p(y, f) = \sum_n [y_n \log \sigma(f_n) + (1-y_n)\log(1-\sigma(f_n))] - \frac{1}{2}f^T K^{-1} f$$


<span style="color: red;">

The log joint splits as log-likelihood (first sum, Bernoulli) + log-prior (second term, Gaussian with covariance $K$ and mean $0$).

</span>

Gradient and Hessian (needed for MAP + covariance):
$$\nabla_f \log p(y,f) = g - K^{-1}f, \quad g_n = y_n - \sigma(f_n)$$
$$\nabla_f^2 \log p(y,f) = -\Lambda - K^{-1}, \quad \Lambda_{nn} = \sigma(f_n)(1-\sigma(f_n))$$

Laplace approximation:

<span style="color: blue;">

$$q(f) = \mathcal{N}(f \mid \hat{f},\; S), \quad S = (K^{-1} + \Lambda)^{-1}$$


</span>

> **Notation note:** $m = \hat{f}$ is used interchangeably for the posterior mean vector. Slides use $\hat{f}$, weekly notebooks use $m$. Same thing.

> **Note:** $K^{-1}$ here (not $(K+\sigma^2 I)^{-1}$) because GP classification 

> has no observation noise — the likelihood is Bernoulli, not Gaussian. 

> Compare with GP regression where noise $\sigma^2$ is added to the diagonal.

**STEP 2 is to compute the posterior distribution $p(f^*|y, x^*)$ using laplace approximation $q(f)$**

<span style="color: blue;">

$$p(f^*|y, x^*) \approx \mathcal{N}(f^* \mid \mu_{f^*},\; \sigma^2_{f^*})$$

$$\mu_{f^*} = k_*^T K^{-1} m$$

$$\sigma^2_{f^*} = c - k_*^T K^{-1}(K - S)K^{-1}k_*$$

</span>

<span style="color: red;">

Where $m=\hat{f}$ is the MAP estimate from Step 1, and $S$ is the Laplace covariance from Step 1. These are all inputs to Step 2, and not new quantities we need to compute.

</span>


Where:

| Symbol | Meaning | Shape |
|---|---|---|
| $m = \hat{f}$ | Laplace posterior mean from Step 1 | $(N,)$ |
| $S$ | Laplace posterior covariance from Step 1 | $(N \times N)$ |
| $k_*$ | Cross-covariance: $k_*^{(n)} = k(x^*, x_n)$ | $(N,)$ |
| $K$ | Training kernel matrix: $K_{ij} = k(x_i, x_j)$ | $(N \times N)$ |
| $c = k(x^*,x^*)$ | Prior variance at test point | scalar |

**Notation confusion:** some sources write $k$ instead of $k_*$ for the cross-covariance vector. In the formula $\sigma^2_{f^*} = k - k^T K^{-1}(K-S)K^{-1}k$, the standalone $k$ at the start is actually the scalar $c = k(x^*,x^*)$, NOT the vector. This is a known typo/ambiguity in the slides.

> **Two equivalent variance formulas** we may see where both correct:

> Slides/weekly: $\sigma^2_{f^*} = c - k_*^T K^{-1}(K-S)K^{-1}k_*$

>  Alternative: $\sigma^2_{f^*} = k(x^*,x^*) - k_*^T(K + \Lambda^{-1})^{-1}k_*$

**In Python:**
```python
k_star = kernel(x_star, x)               # shape (N,)
K_mat  = kernel(x[:, None], x[None, :])  # shape (N, N)
c      = kernel(x_star, x_star)          # scalar

mu_f  = k_star @ jnp.linalg.solve(K_mat, m)
v     = jnp.linalg.solve(K_mat, k_star)
var_f = c - v @ (K_mat - S) @ v
```


**STEP 3 is to compute $p(y^*=1|y, x^*)$ via probit approximation**

The integral $\int \sigma(f^*) p(f^*|y,x^*) df^*$ is intractable, so use probit:

<span style="color: blue;">

$$p(y^*=1|y, x^*) \approx \Phi\!\left(\frac{\mu_{f^*}}{\sqrt{\frac{8}{\pi} + \sigma^2_{f^*}}}\right)$$

</span>

> **Notation warning:** we may see this written as $\sqrt{1 + \frac{\pi}{8}\sigma^2_{f^*}}$ in some places. These are **different** — the weekly notebook uses $\frac{8}{\pi}$ in the denominator, the slides sometimes use $1 + \frac{\pi}{8}$. The weekly notebook version is correct for this course.


$$p(y^*=0|y, x^*) = 1 - p(y^*=1|y, x^*)$$


**In Python:**
```python
from scipy.stats import norm
p_y_star = norm.cdf(mu_f / jnp.sqrt(8/jnp.pi + var_f))
```


**Prior predictive (no data):**

$$p(y^*=1|x^*) = \Phi\!\left(\frac{0}{\sqrt{\frac{8}{\pi} + k(x^*,x^*)}}\right) = 0.5$$

Always 0.5 when prior mean is zero, regardless of kernel.

##### **6.3 Constructing valid kernels**

If $k_1$ and $k_2$ are valid kernels, so are:

<span style="color: blue;">

> $k_1 + k_2$ (sum)

> $k_1 \cdot k_2$ (product)

> $c \cdot k_1$ for $c > 0$ (scaling)

</span>

Requirements are that they must be symmetric and positive semi-definite.


##### **6.4 Kernels and feature spaces (Mercer's theorem)**


Every valid kernel $k(x,x')$ corresponds to an inner product in some (possibly infinite-dimensional) feature space $\phi(x)$:

<span style="color: blue;">

$$k(x,x') = \phi(x)^T\phi(x')$$

</span>

The SE kernel has an **infinite-dimensional** implicit feature space.

##### **6.5 Neural networks**

Example of a two-layer neural network (NN) with a single output:
$$
z_1 = h_1(W_1x+b_1)
$$
$$
z_2 = h_2(W_2 z_1 + b_2)
$$
$$
f=W_3 z_2 + b_3
$$

From input to output with the bias term left out, we have:
$$
f(x)=W_3h_2(W_2 h_1(W_1 x))
$$
We use the linear model with basis functions from week 2:
$$
f(x)=w^T \phi(x)
$$
Now the linear model with adaptive basis functions:
$$
z_2(x)=h_2(W_2h_1(W_1 x))=\phi_W(x)
$$
$$
f(x)=W_3z_2=W_3\phi_W(x)
$$
Where the neural networks probabilisitc modelling is:
$$
p(y_n|w) = \mathcal{N}(y_n|f(x|w), \sigma^2)
$$

For **Bayesian inference** over $w$, the same pipeline applies as logistic regression:

> Place prior $p(w) = \mathcal{N}(w|0, \alpha^{-1}I)$

> Posterior $p(w|y)$ is intractable (non-linear activations break conjugacy)

> Use Laplace approximation: find $w_\text{MAP}$, compute Hessian, approximate posterior as Gaussian